<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02: Preprocesamiento y Análisis de Datos (Landslide4Sense)
**Proyecto:** Detección de Deslizamientos mediante Inteligencia Artificial  
**Institución:** Universidad EAFIT  

Este notebook unifica la conexión a Drive, el análisis de balance de clases y el estudio de índices espectrales.

## 1. Conexión y Configuración de Rutas
Montamos Drive y localizamos los archivos .h5 de forma recursiva.

In [ ]:
from google.colab import drive
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

drive.mount('/content/drive')

base_path = Path('/content/drive/MyDrive/Landslide4Sense')
detectar_img = list(base_path.glob('**/TrainData/img'))

if detectar_img:
    train_img_dir = detectar_img[0]
    train_mask_dir = detectar_img[0].parent / "mask"
    img_list = sorted(list(train_img_dir.glob("*.h5")))
    mask_list = sorted(list(train_mask_dir.glob("*.h5")))
    print(f"✅ Éxito! Datos encontrados en: {train_img_dir}")
    print(f"Imágenes: {len(img_list)} | Máscaras: {len(mask_list)}")
else:
    print("❌ ERROR: No se encontró la estructura de carpetas.")

## 2. Análisis Visual Multiespectral
Visualización de bandas Sentinel-2 y capas geotécnicas.

In [ ]:
def load_h5(file_path):
    with h5py.File(file_path, 'r') as f:
        return np.array(f[list(f.keys())[0]])

if len(img_list) > 0:
    idx = np.random.randint(0, len(img_list))
    img = load_h5(img_list[idx])
    mask = load_h5(mask_list[idx])
    
    rgb = img[:, :, [3, 2, 1]]
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    
    fig, ax = plt.subplots(1, 4, figsize=(22, 5))
    ax[0].imshow(rgb); ax[0].set_title("Sentinel-2 (RGB)"); ax[0].axis('off')
    ax[1].imshow(img[:,:,12], cmap='terrain'); ax[1].set_title("Pendiente (Slope)"); ax[1].axis('off')
    ax[2].imshow(img[:,:,13], cmap='gist_earth'); ax[2].set_title("Elevación (DEM)"); ax[2].axis('off')
    ax[3].imshow(mask, cmap='magma'); ax[3].set_title("Zonas de Deslizamiento"); ax[3].axis('off')
    plt.show()

## 3. Estadísticas de Clases
Cálculo del desbalance para el reporte de tesis.

In [ ]:
def check_balance(limit=50):
    clase_0, clase_1 = 0, 0
    for i in range(min(limit, len(mask_list))):
        m = load_h5(mask_list[i])
        clase_1 += np.sum(m == 1)
        clase_0 += np.sum(m == 0)
    total = clase_0 + clase_1
    print(f"Análisis de balance (píxeles):")
    print(f"- Sin deslizamiento: {clase_0/total:.2%}")
    print(f"- Con deslizamiento: {clase_1/total:.2%}")

check_balance()

## 4. Análisis Avanzado: Índices Espectrales e Histogramas
Estudio de NDVI y rangos de bandas para optimizar el entrenamiento.

In [ ]:
def analyze_features(idx):
    img = load_h5(img_list[idx])
    red = img[:,:,3]
    nir = img[:,:,7]
    ndvi = (nir - red) / (nir + red + 1e-8)
    
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    im1 = ax[0].imshow(ndvi, cmap='RdYlGn')
    ax[0].set_title("Índice de Vegetación (NDVI)")
    plt.colorbar(im1, ax=ax[0])
    
    ax[1].hist(img[:,:,12].ravel(), bins=50, color='brown', alpha=0.7)
    ax[1].set_title("Distribución de Pendientes")
    
    ax[2].boxplot([img[:,:,i].ravel() for i in range(4)], labels=['B1','B2','B3','B4'])
    ax[2].set_title("Rangos de Valores por Banda")
    plt.show()

if len(img_list) > 0:
    analyze_features(np.random.randint(0, len(img_list)))

### Conclusión del Preprocesamiento
1. **Desbalance:** Se confirma un desbalance severo (aprox 2.7% positivos). Se usará **Focal Loss**.
2. **NDVI:** El índice muestra una clara transición en zonas de suelo desnudo, lo cual es prometedor para el modelo.